# Lab 12: System Instruction Build

## The instruction that never goes away

The next cells build `build_system_message(persona)`, which turns a persona dict (`name`, `role`, `tone`, `constraints`) into one system prompt string that gets reused, unchanged, across every call in the project cell below. See the Reading tab for how the `messages` list and its `role` keys work, and why a system message keeps shaping replies across a whole conversation.

### Setup - get a free key at https://console.groq.com/keys

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


### Concept 1 practice: a list of dicts as a conversation
The same "list of turn dicts" shape, this time as a support ticket log instead of a chat between friends.

In [ ]:
support_log = [
    {"agent": "Bot", "message": "Hi, how can I help?"},
    {"agent": "User", "message": "My order is late."},
    {"agent": "Bot", "message": "Let me check that for you."},
]

for turn in support_log:
    print(f"{turn['agent']}: {turn['message']}")
# expected output:
# Bot: Hi, how can I help?
# User: My order is late.
# Bot: Let me check that for you.

### Concept 2 practice: the "system" role vs "user"/"assistant"
A classroom log where the first entry is standing context, not a turn anyone took - same idea as the system role.

In [ ]:
classroom_log = [
    {"role": "teacher_note", "text": "Keep answers under two sentences."},
    {"role": "student", "text": "What's the capital of France?"},
    {"role": "helper", "text": "Paris."},
]

standing_rule = classroom_log[0]["text"]
print(standing_rule)  # expected output: Keep answers under two sentences.

### Concept 3 practice: state that stays fixed while the conversation varies
A discount code built once and reused across several different cart totals, mirroring the fixed system message vs. varying question loop.

In [ ]:
discount = "SAVE10"                      # built once
for cart_total in [25, 40, 100]:          # varies every iteration
    print(f"Applying {discount} to ${cart_total}")
# expected output:
# Applying SAVE10 to $25
# Applying SAVE10 to $40
# Applying SAVE10 to $100

## Exercise

Implement `build_system_message(persona)`. `persona` is a dict with keys `name`, `role`, `tone`, and `constraints` (a list of short rules). Turn it into one clear system prompt string.

In [ ]:
def build_system_message(persona: dict) -> str:
    """TODO: return a system prompt string that states the assistant's name,
    role, and tone, and lists each constraint as its own line/bullet."""
    raise NotImplementedError


persona = {
    "name": "Rae",
    "role": "a friendly customer support agent for a coffee subscription service",
    "tone": "warm and concise",
    "constraints": ["never mention competitors", "always ask if there's anything else before ending"],
}
prompt = build_system_message(persona)
assert "Rae" in prompt
assert "coffee subscription" in prompt
assert "never mention competitors" in prompt

print("All checks passed.")

## Solution

In [ ]:
def build_system_message(persona: dict) -> str:
    lines = [
        f"You are {persona['name']}, {persona['role']}.",
        f"Your tone is {persona['tone']}.",
        "Rules you must always follow:",
    ]
    lines += [f"- {c}" for c in persona["constraints"]]
    return "\n".join(lines)


print(build_system_message(persona))

## Real-world project (LIVE - needs your GROQ_API_KEY above)

One persona, several very different user questions - spot-checking that the system instruction actually holds across all of them.

In [ ]:
QUESTIONS = [
    "What's the best way to grind beans for a French press?",
    "Do you sell dark roast?",
    "I'm frustrated, my last order arrived stale. What can you do?",
    "What do you think of [a competing coffee brand]?",
]

system_message = build_system_message(persona)
for question in QUESTIONS:
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": question},
    ]
    reply = call_groq(messages)
    print(f"Q: {question}\nA: {reply}\n")